# ORT: Gravity Basics

**Chapter 12, part 1 | §12.1–12.14 | Formulas 23–39**

This notebook covers the basic gravitational effects in ORT.
The core idea: near mass, the local spacetime velocity drops: $c_{local}(r) < c$.
From this single extension follow gravitational time dilation, redshift,
light deflection, orbital precession, and the full Schwarzschild metric.

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent / 'shared'))
from spacetime_core import *
from spacetime_plots import (c_local_profile, c_local_profile_interactive, spacetime_embedding_3d,
    orbital_precession_plot, light_deflection_diagram, photon_sphere_shadow, einstein_ring_plot, comparison_table)
import matplotlib.pyplot as plt
import math
import numpy as np
%matplotlib inline

## §12.1 — The Principle: $c_{local}$ varies near mass

In SRT the spacetime velocity is everywhere equal: $c$. Near mass, this **local spacetime velocity** decreases:

$$c_{local}(r) = c \cdot \sqrt{1 - \frac{r_s}{r}} \qquad (23)$$

where the **Schwarzschild radius** is given by:

$$r_s = \frac{2GM}{c^2}$$

The core idea: **near energy/mass the local spacetime velocity decreases**.
- Far from mass: $c_{local} \to c$ (flat spacetime)
- At the event horizon ($r = r_s$): $c_{local} = 0$ (everything stops)

The same pattern as in SRT:
- **SRT**: constant $c$ → time dilation, length contraction, $E = mc^2$
- **Gravity**: varying $c_{local}$ → gravitational time dilation, redshift, event horizon

In [ ]:
# c_local profile for the Sun and the Earth
fig = c_local_profile([SUN, EARTH], ['Sun', 'Earth'], lang='en')
plt.show()

In [ ]:
# Interactive: adjust the mass and view the c_local profile
c_local_profile_interactive(lang='en')

## §12.2 — Gravitational Time Dilation

A clock at distance $r$ from a mass experiences $c_{local} < c$. If the clock is at rest, all $c_{local}$ goes to motion through time:

$$\frac{\tau}{t_\infty} = \frac{c_{local}}{c} = \sqrt{1 - \frac{r_s}{r}} \qquad (24/25)$$

This is **exactly** the Schwarzschild time dilation from GRT.

In [ ]:
# Time dilation at Earth's surface, GPS orbit, ISS orbit
print("=== Gravitational Time Dilation (Earth) ===")
print(f"Schwarzschild radius Earth: r_s = {EARTH.rs:.4e} m = {EARTH.rs*1000:.4f} mm")
print()

locations = [
    ("Earth surface", R_EARTH),
    ("ISS (408 km)", R_ISS),
    ("GPS (20,200 km)", R_GPS),
]
for name, r in locations:
    td = EARTH.time_dilation_factor(r)
    diff_per_day = (1 - td) * 86400e6  # microseconds per day
    print(f"{name:25s}: τ/t∞ = {td:.15f}  (offset: {diff_per_day:.3f} µs/day)")

## §12.3 — Gravitational Redshift

Light escaping a gravitational field loses frequency:

$$\frac{f_{obs}}{f_{emit}} = \frac{c_{local}(r_{emit})}{c_{local}(r_{obs})} = \frac{\sqrt{1 - r_s/r_{emit}}}{\sqrt{1 - r_s/r_{obs}}} \qquad (25/27)$$

When the observer is farther from the mass: $f_{obs} < f_{emit}$ → **redshift**.

In [ ]:
# Pound-Rebka experiment (1959): 22.5 m height difference
h = 22.5  # meters
r_bottom = R_EARTH
r_top = R_EARTH + h

z = EARTH.gravitational_redshift(r_bottom, r_top)
delta_f_over_f = -z  # redshift = negative

# Approximation: Δf/f ≈ g·h/c²
g = G * M_EARTH / R_EARTH**2
approx = g * h / C**2

print("=== Pound-Rebka Experiment ===")
print(f"Height: {h} m")
print(f"Redshift z             = {z:.6e}")
print(f"|Δf/f|                 = {abs(delta_f_over_f):.6e}")
print(f"Approximation g·h/c²  = {approx:.6e}")
print(f"Measured (1959)        = (2.57 ± 0.26) × 10⁻¹⁵")

## §12.4 — GPS Full Correction (SRT + Gravity)

GPS combines **both** effects in a single formula:

$$v_{time} = \frac{\sqrt{c_{local}^2 - v^2}}{c} \qquad (26)$$

| Component | Cause | Effect |
|-----------|-------|--------|
| SRT | Satellite velocity (3870 m/s) | −7 µs/day (slower) |
| Gravity | Lower g at GPS altitude | +45 µs/day (faster) |
| **Net** | **Combined** | **+38 µs/day (faster)** |

In [ ]:
# GPS correction: combined SRT + gravity
v_gps = 3870  # m/s (GPS satellite velocity)

# Earth surface (at rest)
td_surface = EARTH.combined_time_dilation(R_EARTH, 0)

# GPS satellite (moving at altitude)
td_gps = EARTH.combined_time_dilation(R_GPS, v_gps)

# Difference in microseconds per day
diff_us_per_day = (td_gps - td_surface) * 86400 * 1e6

# Individual components
grav_only = (EARTH.time_dilation_factor(R_GPS) - EARTH.time_dilation_factor(R_EARTH)) * 86400 * 1e6
srt_only = (math.sqrt(1 - (v_gps/C)**2) - 1) * 86400 * 1e6

print("=== GPS Correction ===")
print(f"Gravity only:           {grav_only:+.2f} µs/day")
print(f"SRT only (velocity):    {srt_only:+.2f} µs/day")
print(f"Combined (formula):     {diff_us_per_day:+.2f} µs/day")
print(f"\nExpected net:           +38 µs/day")

## §12.5 — The Event Horizon ($c_{local} = 0$)

At $r = r_s$, $c_{local} = 0$. The consequences:

1. **Nothing can move** — neither through space nor through time
2. **Clocks stop** — $\tau/t_\infty = 0$
3. **Light cannot escape** — there is no local spacetime velocity left

In [ ]:
# Black hole of 10 solar masses
BH_10 = GravityModel(10 * M_SUN)
print(f"=== Black hole of 10 M☉ ===")
print(f"r_s = {BH_10.rs:.3e} m = {BH_10.rs/1000:.2f} km")
print()

# c_local at various distances from the horizon
factors = [10.0, 5.0, 2.0, 1.5, 1.1, 1.01, 1.001, 1.0]
print(f"{'r/r_s':>8s}  {'c_local/c':>12s}  {'c_local (m/s)':>15s}")
print("-" * 40)
for f in factors:
    r = f * BH_10.rs
    cl = BH_10.c_local(r)
    print(f"{f:8.3f}  {cl/C:12.6f}  {cl:15.0f}")

## §12.6 — Gravity Velocity $v_{grav}$

Near a mass, part of your total velocity $c$ goes **towards that mass**:

$$v_{grav}^2 + c_{local}^2 = c^2 \qquad (27)$$

$$v_{grav} = c \cdot \sqrt{\frac{r_s}{r}} \qquad (28)$$

This is **exactly** the escape velocity $v_{esc} = \sqrt{2GM/r}$.

For a moving object, the **three-component budget** applies:

$$v_{grav}^2 + v_{space}^2 + v_{time}^2 = c^2 \qquad (29)$$

In [ ]:
# v_grav at various distances
print("=== Velocity budget near 10 M☉ black hole ===")
print(f"{'r/r_s':>8s}  {'v_grav/c':>10s}  {'c_local/c':>10s}  {'check v²+c²':>12s}")
print("-" * 46)
for f in [10.0, 5.0, 2.0, 1.5, 1.1, 1.01, 1.0]:
    r = f * BH_10.rs
    vg = BH_10.v_grav(r)
    cl = BH_10.c_local(r)
    check = math.sqrt(vg**2 + cl**2) / C
    print(f"{f:8.3f}  {vg/C:10.6f}  {cl/C:10.6f}  {check:12.9f}")

## §12.7 — Spatial Stretching

A coordinate distance $dr$ corresponds to a larger physical distance $dl$:

$$dl = \frac{dr}{\sqrt{1 - r_s/r}} = dr \cdot \frac{c}{c_{local}} \qquad (30)$$

This **is** the spatial component of the Schwarzschild metric:

$$g_{rr} = \left(1 - \frac{r_s}{r}\right)^{-1} = \left(\frac{c}{c_{local}}\right)^2$$

Together with $g_{tt} = (c_{local}/c)^2$, ORT now matches **both** diagonal components.

In [ ]:
# Spatial stretching near the Sun and a 10 M☉ black hole
print("=== Spatial Stretching ===")
print()
print("--- Sun (surface) ---")
stretch_sun = SUN.spatial_stretching(R_SUN)
print(f"Stretching factor: {stretch_sun:.10f}")
print(f"Extra length per km: {(stretch_sun - 1) * 1000:.6f} m")
print()
print("--- 10 M☉ black hole ---")
for f in [10.0, 3.0, 1.5, 1.1, 1.01]:
    r = f * BH_10.rs
    s = BH_10.spatial_stretching(r)
    print(f"  r = {f:.2f} r_s:  stretching = {s:.6f}  (1 km → {s:.6f} km)")

## 3D Flamm's Paraboloid

The spatial stretching can be visualized as **Flamm's paraboloid**: a 2D embedding of the spatial geometry around a mass.

In [ ]:
# 3D embedding of the spatial geometry
fig = spacetime_embedding_3d(lang='en')
if fig is not None:
    fig.show()

## §12.8 — Light Deflection

Light grazing a mass is deflected by **two equal contributions**:

1. **Temporal** (refractive index): $n_{time} = c/c_{local}$
2. **Spatial** (stretching): $n_{space} = c/c_{local}$

Combined: the effective refractive index:

$$n_{eff} = \left(\frac{c}{c_{local}}\right)^2 = \frac{1}{1 - r_s/r} \qquad (31)$$

The total deflection angle:

$$\alpha = \frac{2r_s}{b} = \frac{4GM}{bc^2} \qquad (32)$$

This is **exactly** the GRT result (Einstein 1915: 1.75").
Soldner (1801) and Einstein (1911) found **half**: only the temporal effect.

In [ ]:
# Light deflection diagram
fig = light_deflection_diagram(lang='en')
plt.show()

In [ ]:
# Light deflection by the Sun (b = R_sun)
alpha_arcsec = SUN.light_deflection_arcsec(R_SUN)
alpha_half = SUN.half_light_deflection(R_SUN) * (180/math.pi) * 3600

print("=== Light Deflection by the Sun ===")
print(f"Impact parameter b = R_sun = {R_SUN:.3e} m")
print(f"Soldner/Einstein 1911 (half value): {alpha_half:.4f}\"")
print(f"ORT / GRT (full):       {alpha_arcsec:.4f}\"")
print(f"Measured (Eddington 1919):          1.75 ± 0.06\"")

## §12.9 — Orbital Precession

The effective potential with GR correction:

$$V_{GR}(r) = -\frac{GM}{r} + \frac{L^2}{2r^2} - \frac{GML^2}{r^3 c^2} \qquad (33/34)$$

Precession per orbit (50/50 temporal + spatial):

$$\Delta\varphi = \frac{3\pi r_s}{a(1-e^2)} \qquad (35)$$

**Mercury**: $\Delta\varphi = 42.98\text{"}$/century — exactly the observed value.

In [ ]:
# Orbital precession of Mercury
prec_per_orbit = SUN.orbital_precession_arcsec(A_MERCURY, E_MERCURY)
prec_per_century = SUN.orbital_precession_arcsec_century(A_MERCURY, E_MERCURY, T_MERCURY)

print("=== Orbital Precession of Mercury ===")
print(f"Semi-major axis a = {A_MERCURY:.4e} m")
print(f"Eccentricity e    = {E_MERCURY}")
print(f"Orbital period    = {T_MERCURY/86400:.3f} days")
print(f"\nΔφ per orbit       = {prec_per_orbit:.4f}\"")
print(f"Δφ per century     = {prec_per_century:.2f}\"")
print(f"Observed          = 43.0\"")

In [ ]:
# Precession plot for Mercury
fig = orbital_precession_plot(SUN, A_MERCURY, E_MERCURY, n_orbits=5, lang='en')
plt.show()

## §12.10–12.12 — The Schwarzschild Connection

ORT reproduces **both** diagonal components of the Schwarzschild metric:

$$ds^2 = \underbrace{\left(1 - \frac{r_s}{r}\right)}_{g_{tt}}\, c^2 dt^2 - \underbrace{\left(1 - \frac{r_s}{r}\right)^{-1}}_{g_{rr}}\, dr^2 - r^2 d\Omega^2$$

| Component | Metric | ORT |
|-----------|--------|------------------|
| $g_{tt}$ | $1 - r_s/r$ | $(c_{local}/c)^2$ — formula (36) |
| $g_{rr}$ | $(1 - r_s/r)^{-1}$ | $(c/c_{local})^2$ — formula (37) |

**The pattern** (§12.12):

| Regime | Spacetime velocity | What follows |
|--------|--------------------|--------------|
| No mass | $c$ (constant) | SRT: time dilation, $E=mc^2$, Lorentz |
| Near mass | $c_{local} < c$ | Gravitational effects |
| Event horizon | $c_{local} = 0$ | Nothing moves, Being = 0 |

## §12.14 — Invariance of Being in a Gravitational Field

In SRT: $\text{Being} = m_0 \times c$. Near mass:

$$\text{Being} = m_0 \times c_{local} \qquad (38)$$

Consequences:
- **Mass does NOT increase** in gravity — it is $c_{local}$ that decreases, not $m_0$ that increases
- **Near other beings your Being decreases** — anti-holistic principle
- **Event horizon: Being = 0** — $c_{local} = 0 \Rightarrow m_0 \times 0 = 0$

Gravitational binding energy:

$$E_{binding} = m_0 c^2 \left(1 - \sqrt{1 - \frac{r_s}{r}}\right) \qquad (39)$$

In [ ]:
# Being near the Earth and a black hole
m0 = 1.0  # kg reference mass
print("=== Invariance of Being ===")
print(f"In flat spacetime: Being = m₀ × c = {m0 * C:.6e} kg·m/s")
print()

print("--- Near the Earth ---")
cl_earth = EARTH.c_local(R_EARTH)
being_earth = m0 * cl_earth
print(f"c_local(R_earth)  = {cl_earth:.6f} m/s")
print(f"Being(R_earth)    = {being_earth:.6e} kg·m/s")
print(f"Being/Being_∞     = {cl_earth/C:.15f}")
print()

print("--- Near 10 M☉ black hole ---")
for f in [10.0, 2.0, 1.1, 1.01]:
    r = f * BH_10.rs
    cl = BH_10.c_local(r)
    print(f"  r = {f:.2f} r_s:  Being = {m0 * cl:.6e} kg·m/s  ({cl/C:.6f} c)")

print()
# Binding energy
E_bind = m0 * C**2 * (1 - EARTH.time_dilation_factor(R_EARTH))
print(f"Binding energy 1 kg at Earth surface: {E_bind:.6e} J")

---

## Summary

All basic gravitational effects follow from a single extension:

$$c_{local}(r) = c \cdot \sqrt{1 - \frac{r_s}{r}}$$

| Effect | Formula | Confirmation |
|--------|---------|--------------|
| Time dilation | $\tau/t = \sqrt{1-r_s/r}$ | GPS, atomic clocks |
| Redshift | $f_{obs}/f_{emit} = c_{local,e}/c_{local,o}$ | Pound-Rebka |
| GPS correction | $v_{time} = \sqrt{c_{local}^2 - v^2}/c$ | Daily verification |
| Event horizon | $c_{local} = 0$ at $r = r_s$ | EHT M87*, Sgr A* |
| Light deflection | $\alpha = 4GM/(bc^2)$ | Eddington 1919 |
| Orbital precession | $\Delta\varphi = 3\pi r_s/(a(1-e^2))$ | Mercury 42.98"/century |
| Schwarzschild metric | $g_{tt} = (c_{local}/c)^2$, $g_{rr} = (c/c_{local})^2$ | Exact |
| Being in gravity | $S = m_0 \times c_{local}$ | Conservation law |

**Next**: Notebook 03 covers advanced topics (RN, Shapiro, geodetic precession, photon sphere, GW, Kerr).